# Case 9: Data quality failure

*Hands-On Electroretinography in the Age of AI*, Chapter 19, §19.9.

Mr. G., 71, has known intermediate AMD and attends for a repeat ERG. This case produces no clinical result. It shows the two safe failure modes of the pipeline, using the pipeline and server code:

* **Scenario A, SIGNAL_QUALITY_WARNING:** a skin electrode recording with an SNR of 2.1.
* **Scenario B, ELECTRODE_UNKNOWN:** a CSV file with a blank `electrode_type` field, which halts processing.

The values are constructed for teaching; they are not patient data.

**Setup.** In Google Colab, the first cell clones the repository. Elsewhere, run the notebook from the repository.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('/content/ERG-Analysis-API'):
    !git clone -q https://github.com/AI-Fairness-com/ERG-Analysis-API.git /content/ERG-Analysis-API
sys.path.insert(0, '/content/ERG-Analysis-API/examples')
sys.path.insert(0, os.path.abspath('examples') if os.path.isdir('examples') else os.getcwd())
from case_helpers import zone_table, check, REPORT, erg

**Scenario A: the skin electrode thresholds (Chapter 2 §2.3.4).** Pass at 3 or above, Warning from 2.0 to 2.9, Fail below 2.0. An SNR of 2.1 is in the Warning band.

In [2]:
for snr in (3.0, 2.1, 2.0, 1.9):
    status, note = erg.CONFIG.get_electrode_snr_status('skin', snr)
    print(f'SNR {snr:.1f}: {status:7s} {note}')

SNR 3.0: PASS    SNR 3.0 ≥ 3.0 threshold
SNR 2.1: WARNING SNR 2.1 is below PASS threshold (3.0)
SNR 2.0: WARNING SNR 2.0 is below PASS threshold (3.0)
SNR 1.9: FAIL    SNR 1.9 is below WARNING threshold (2.0)


**No zone for a skin electrode.** The reference limits are the silver thread limits of Baker et al. (2025), so a skin recording receives no zone and no Traffic Light. The pipeline returns UNAVAILABLE.

In [3]:
rows = REPORT.classify_parameters({'b_amp_uv': 52, 'b_implicit_ms': 30.4}, 'skin', 'LA 3.0', 71)
print([(r['parameter'], r['zone']) for r in rows if r['parameter'].startswith('b_')])
tl = REPORT.composite_traffic_light(rows, 'skin')
print('Traffic Light:', tl['signal'])
print(tl['message'])

[('b_amp', 'UNAVAILABLE'), ('b_imp', 'UNAVAILABLE')]
Traffic Light: UNAVAILABLE
Traffic Light zones are available for silver thread (DTL fiber) electrodes, and for gold foil electrodes after conversion to silver thread equivalents, using the age-specific reference limits of Baker et al. (2025). No reference limits are held here for contact lens or skin electrodes. The measured values are reported, and interpretation needs laboratory-specific reference limits (ISCEV 2022; Davis and Hamilton, 2021).


**Warning mode does not halt (§19.9.1).** The classification continues and returns the Stage 2 probabilities, marked for reference only; the SHAP attribution is not shown, because it may reflect noise. The Layer 3 Specialist Report carries a gray badge with the note below.

In [4]:
probabilities = {'AMD': 0.49, 'Cone Dystrophy': 0.26, 'Glaucoma': 0.12, 'DR': 0.08, 'Other two classes': 0.05}
assert round(sum(probabilities.values()), 2) == 1.0
print('Stage 2 probabilities (for reference only):')
for name, p in probabilities.items():
    print(f'  {name:18s} {p:.2f}')
print('SHAP attribution: not displayed (signal quality warning)')
print('Badge: “Result unreliable: signal quality warning. Do not use for clinical '
      'decision-making without a repeat recording.”')

Stage 2 probabilities (for reference only):
  AMD                0.49
  Cone Dystrophy     0.26
  Glaucoma           0.12
  DR                 0.08
  Other two classes  0.05
SHAP attribution: not displayed (signal quality warning)
Badge: “Result unreliable: signal quality warning. Do not use for clinical decision-making without a repeat recording.”


**Scenario B: blank electrode type (§19.9.2).** The electrode type has no default, because the SNR thresholds and the reference limits both depend on it. The server takes it from the form or, if that is blank, from the CSV's `electrode_type` column (Appendix A). Here both are blank, so the API halts before any signal processing and returns the Layer 1 error report. The request is sent to the server code locally, through Flask's test client.

In [5]:
import contextlib, io
import pandas as pd
with contextlib.redirect_stdout(io.StringIO()):
    import erg_server
client = erg_server.app.test_client()

csv = pd.read_csv('data/samples/normal_001_DA3.csv')
csv['electrode_type'] = ''
reply = client.post('/api/v1/analyse', content_type='multipart/form-data', data={
    'file': (io.BytesIO(csv.to_csv(index=False).encode()), 'mr_g.csv'),
    'age_group': '≥60y', 'protocol': 'DA 3.0'})
report = reply.get_json()
print('HTTP status:', reply.status_code)
print('Error:', report['error'], '| missing field:', report['missing_field'])
print('Valid values:', ', '.join(report['valid_values']))
print()
print(report['message'])

HTTP status: 400
Error: ELECTRODE_UNKNOWN | missing field: electrode_type
Valid values: contact_lens, gold_foil, dtl_fiber, skin

PROCESSING HALTED: The required field electrode_type is missing from the submitted CSV file. The electrode type is required for: (1) signal quality threshold assignment; (2) normative reference limit selection. Processing cannot continue without this field. No classification has been attempted. Action required: resubmit the CSV file with the electrode_type field completed. Valid values: contact_lens, gold_foil, dtl_fiber, skin. See Appendix A for the full CSV field specification.
